# 02 Datasets Metadata And Lazy Loading

`OctoSense.datasets` is the dataset-level module in OctoSense's
`Unified Data Abstraction`. It solves the problem that RF datasets
are large, metadata-heavy, and difficult to inspect if every
operation has to load raw tensors first. The design uses
machine-readable metadata together with view-based operations, so
users can inspect labels, tasks, and splits first and only load RF
samples when they are actually needed. This notebook shows that
result on the real Widar3 sample under
`demo/notebook/data/widar3/`: it checks `metas.csv`, verifies the
`gesture_name -> label` mapping, and then loads a small runnable
split mapping.

### Set Up Dataset Utilities And Timing Helpers

Import the downloader and lightweight timing tools needed to prepare
the notebook-local sample root and compare metadata-only work
against later materialization.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = (NOTEBOOK_DIR / "../..").resolve()
SRC_ROOT = REPO_ROOT / "src"
TEST_DATA_ROOT = NOTEBOOK_DIR / "data"
PRESET_ROOT = NOTEBOOK_DIR / "presets"
NOTEBOOK_TREE_DEPTH = 2
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from loguru import logger
import octosense as octo
import csv
import time
from octosense.datasets.storage.downloader import download_dataset


logger.remove()
logger.add(sys.stderr, colorize=False, format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")
logger.info("Notebook setup completed with repository-relative paths")
logger.info("Default describe tree render depth set to {}", NOTEBOOK_TREE_DEPTH)

### Prepare The Local Widar3 Sample Root And Inspect Metadata Views

Reuse or download the notebook-local Widar3 sample, load the public
`DatasetView`, inspect the real metadata CSV, and verify the split
structure and label mapping as the foundation for the later tensor
walkthrough.

In [ ]:
logger.info("Preparing the real Widar3 sample root")
dataset_root = TEST_DATA_ROOT / "widar3"
canonical_sample_path = (
    dataset_root
    / "CSI"
    / "20181211"
    / "user3"
    / "user3-1-1-1-1-r1.dat"
)
if canonical_sample_path.exists():
    logger.info("Using local Widar3 sample root at {}", dataset_root.relative_to(REPO_ROOT))
else:
    logger.info("Downloading Widar3 sample payload into {}", dataset_root.relative_to(REPO_ROOT))
    dataset_root = download_dataset("widar3", root=dataset_root, part="sample", force=False)
    logger.info("Using downloaded Widar3 sample root at {}", dataset_root.relative_to(REPO_ROOT))
metadata_csv_path = dataset_root / "metas.csv"
if not metadata_csv_path.exists():
    raise FileNotFoundError(f"Expected Widar3 metadata CSV at {metadata_csv_path}")

with metadata_csv_path.open(encoding="utf-8") as handle:
    metadata_csv_rows = list(csv.DictReader(handle, delimiter="|"))

load_started = time.perf_counter()
local_view = octo.datasets.load(
    "widar3",
    modalities=["wifi"],
    variant="sample",
    split_scheme="sample",
    task_binding="gesture",
    path=str(dataset_root),
)
load_elapsed = time.perf_counter() - load_started
view_ops_started = time.perf_counter()
local_rows = local_view.metadata_rows()
train_view = local_view.get_split("train")
test_view = local_view.get_split("test")
split_names = tuple(sorted({str(row["split"]) for row in local_rows}))
view_ops_elapsed = time.perf_counter() - view_ops_started
real_labels = sorted(
    {
        (row["gesture_name"], row["label"])
        for row in local_rows
    }
)
preview_rows = []
seen_splits = set()
for row in local_rows:
    split_name = str(row["split"])
    if split_name in seen_splits:
        continue
    preview_rows.append(row)
    seen_splits.add(split_name)
assert len(local_rows) == len(local_view)
assert {"train", "test"}.issubset(set(split_names))

print(metadata_csv_rows[0]["gesture_list"])
print(real_labels)
for row in preview_rows:
    print(f'{row["split"]} | label={row["label"]} | gesture={row["gesture_name"]} | sample_id={row["sample_id"]}')
print(local_view.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
logger.info("Loaded {} samples from {}", len(local_view), dataset_root.relative_to(REPO_ROOT))
logger.info("Metadata CSV location: {}", metadata_csv_path.relative_to(REPO_ROOT))
logger.info("Metadata row count: {}", len(local_rows))
logger.info("DatasetView load took {:.6f}s", load_elapsed)
logger.info("Metadata-only view ops took {:.6f}s", view_ops_elapsed)
logger.info("Available splits: {}", split_names)
logger.info("Real labels: {}", real_labels)
logger.info("Train/test sizes: {}/{}", len(train_view), len(test_view))

### Materialize Samples Only After Metadata Validation

Walk the already-validated dataset view through full sample
materialization on a bounded demo split mapping. The notebook-local
sample root contains 17k+ captures, and the explicit split mapping
keeps the walkthrough compact, multi-split, and representative for
end-to-end materialization.

In [ ]:
materialize_started = time.perf_counter()
materialize_split_mapping = local_view.materialize_split_mapping(max_samples_per_split=32)
materialized_labels = []
first_sample = None
first_label = None
first_meta = None
for split_name, split_view in materialize_split_mapping.items():
    split_rows = split_view.metadata_rows()
    for index in range(len(split_view)):
        sample, label = split_view[index]
        materialized_labels.append(label)
        if first_sample is None:
            first_sample = sample
            first_label = label
            first_meta = split_rows[index]
materialize_elapsed = time.perf_counter() - materialize_started

assert first_sample is not None
assert first_label is not None
assert first_meta is not None
assert len(materialized_labels) == sum(len(view) for view in materialize_split_mapping.values())
print(first_sample.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
logger.info(
    "Bounded demo split mapping materialized all {} selected samples across splits {} in {:.6f}s ({:.6f}s/sample)",
    len(materialized_labels),
    {split_name: len(split_view) for split_name, split_view in materialize_split_mapping.items()},
    materialize_elapsed,
    materialize_elapsed / max(len(materialized_labels), 1),
)
logger.info("First materialized sample label: {} ({})", first_label, first_meta["gesture_name"])